# Remake -- removing correlated features

The correlation heatmap in `remake_train.ipynb` showed several near-duplicate
pairs. This notebook tests whether dropping some of them costs accuracy, and
whether it makes the models easier to explain.

| Set | Features | Dropped |
|---|---|---|
| A | 29 | nothing (current) |
| B | 27 | `sale_quarter`, `building_total_floor_area` |
| C | 27 | `sale_quarter`, `mass_total_floor_area` |
| D | 25 | B + `unemployment_rate_by_sa`, `employment_rate_by_sa` |

Why these:
- `sale_quarter` is calculated from `sale_month` (rho = 0.97) -- no new information.
- The two floor areas measure the same thing (rho = 0.96). B and C test which to keep.
- Unemployment rate tracks the deprivation index (rho = 0.88) and employment
  rate tracks median income (rho = 0.85); NZDep is itself built partly from
  census income and employment data.

All five models (XGBoost, CatBoost, Random Forest, neural network, hedonic)
use the same data, folds and processing as the other remake notebooks
(sections 1--3 copied unchanged). Dropping a column happens after the
per-fold processing, so it does not change the folds or the price trim.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OrdinalEncoder

pd.set_option("display.max_columns", 70)

df = pd.read_parquet("./cache/remake_dataset.parquet")

# TimeSeriesSplit assumes row order IS time order -- sort explicitly rather
# than relying on the order the parquet was written in.
df = df.sort_values("sale_date_parsed").reset_index(drop=True)
print(f"Loaded {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Date range: {df['sale_date_parsed'].min().date()} to {df['sale_date_parsed'].max().date()}")

Loaded 59,708 rows, 33 columns
Date range: 2024-01-01 to 2026-08-18


## 1. Feature groups (the full 29)

In [2]:
ID_COLS = ["valuation_id_concat", "sale_id", "sale_date_parsed"]
TARGET = "sale_price_gross"

CAT_COLS = [
    "property_type", "building_age_indicator", "building_condition_indicator",
    "building_construction_indicator", "noise_category_combination",
]
BOOL_COLS = ["in_flood_plain"]
NUM_COLS = [
    # property
    "land_area", "building_total_floor_area", "mass_total_floor_area",
    "off_street_parking", "mass_garage_under_main_roof", "mass_garage_freestanding",
    # location / spatial
    "latitude", "longitude", "distance_to_cbd", "walk_supermarket_network_m",
    "walk_bus_stop_network_m", "distance_to_school", "area_level_deprivation_index",
    # socioeconomic
    "median_personal_income_by_sa", "employment_rate_by_sa", "unemployment_rate_by_sa",
    "population_counts_by_sa", "population_density",
    # environment
    "distance_to_coastline",
    # market / time
    "building_consents", "sale_year", "sale_month", "sale_quarter",
]
FEATURE_COLS = CAT_COLS + BOOL_COLS + NUM_COLS

unassigned = set(df.columns) - set(FEATURE_COLS) - set(ID_COLS) - {TARGET}
missing = set(FEATURE_COLS) - set(df.columns)
assert not unassigned, f"Columns not assigned to any group: {unassigned}"
assert not missing, f"Feature columns not in dataset: {missing}"

print(f"{len(FEATURE_COLS)} features: {len(CAT_COLS)} categorical, {len(BOOL_COLS)} boolean, {len(NUM_COLS)} numeric")

29 features: 5 categorical, 1 boolean, 23 numeric


## 2. Per-fold processing (same as `remake_train.ipynb`)

In [3]:
def prepare_fold(train_df, test_df):
    """Encode, impute and price-trim one fold. Returns X/y for train and test."""
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    encoder.fit(train_df[CAT_COLS])

    X_train = train_df[FEATURE_COLS].copy()
    X_test = test_df[FEATURE_COLS].copy()
    X_train[CAT_COLS] = encoder.transform(train_df[CAT_COLS])
    X_test[CAT_COLS] = encoder.transform(test_df[CAT_COLS])
    for col in BOOL_COLS:
        X_train[col] = X_train[col].astype(int)
        X_test[col] = X_test[col].astype(int)
    X_train = X_train.astype(float)
    X_test = X_test.astype(float)

    nan_cols = [c for c in NUM_COLS if X_train[c].isna().any() or X_test[c].isna().any()]
    train_medians = X_train[nan_cols].median()
    X_train[nan_cols] = X_train[nan_cols].fillna(train_medians)
    X_test[nan_cols] = X_test[nan_cols].fillna(train_medians)

    y_train = train_df[TARGET].values
    y_test = test_df[TARGET].values
    p1, p99 = np.quantile(y_train, [0.01, 0.99])
    train_keep = (y_train >= p1) & (y_train <= p99)
    test_keep = (y_test >= p1) & (y_test <= p99)

    return {
        "X_train": X_train[train_keep], "y_train": y_train[train_keep],
        "X_test": X_test[test_keep], "y_test": y_test[test_keep],
        "test_idx": test_df.index[test_keep], "price_limits": (p1, p99),
    }

## 3. Build the 5 folds

In [4]:
kf = TimeSeriesSplit(n_splits=5)
folds = []
rows = []
for fold_i, (train_idx, test_idx) in enumerate(kf.split(df), start=1):
    f = prepare_fold(df.iloc[train_idx], df.iloc[test_idx])
    assert not f["X_train"].isna().any().any() and not f["X_test"].isna().any().any()
    folds.append(f)
    rows.append({
        "fold": fold_i,
        "train_period_end": df.iloc[train_idx]["sale_date_parsed"].max().date(),
        "test_period": f"{df.iloc[test_idx]['sale_date_parsed'].min().date()} to {df.iloc[test_idx]['sale_date_parsed'].max().date()}",
        "n_train": len(f["y_train"]), "n_test": len(f["y_test"]),
        "price_p1": f["price_limits"][0], "price_p99": f["price_limits"][1],
    })

fold_summary = pd.DataFrame(rows).set_index("fold")
print(f"Total test rows across folds: {fold_summary['n_test'].sum():,}")
fold_summary

Total test rows across folds: 48,777


,train_period_end,test_period,n_train,n_test,price_p1,price_p99
fold,,,,,,
1,2024-06-19,2024-06-19 to 2024-11-07,9760,9779,450000.0,4512000.0
2,2024-11-07,2024-11-07 to 2025-04-01,19505,9685,450000.0,4229700.0
3,2025-04-01,2025-04-01 to 2025-08-22,29263,9783,430000.0,4261500.0
4,2025-08-22,2025-08-22 to 2026-01-24,39008,9761,433089.1,4219250.0
5,2026-01-24,2026-01-24 to 2026-08-18,48774,9769,430243.6,4200000.0


## 4. Feature sets

In [5]:
VARIANTS = {
    "A (29, all)": [],
    "B (27, drop quarter + building floor)": ["sale_quarter", "building_total_floor_area"],
    "C (27, drop quarter + mass floor)": ["sale_quarter", "mass_total_floor_area"],
    "D (25, B + drop (un)employment)": ["sale_quarter", "building_total_floor_area",
                                        "unemployment_rate_by_sa", "employment_rate_by_sa"],
}


def groups(drop):
    """(categorical, boolean, numeric) columns left after dropping `drop`."""
    return ([c for c in CAT_COLS if c not in drop],
            [c for c in BOOL_COLS if c not in drop],
            [c for c in NUM_COLS if c not in drop])


for name, drop in VARIANTS.items():
    cat, boo, num = groups(drop)
    print(f"{name}: {len(cat) + len(boo) + len(num)} features")

A (29, all): 29 features
B (27, drop quarter + building floor): 27 features
C (27, drop quarter + mass floor): 27 features
D (25, B + drop (un)employment): 25 features


## 5. Models

Same fixed settings as the other remake notebooks:

- **XGBoost / CatBoost / Random Forest** -- as in `remake_train.ipynb`.
- **Neural network** -- as in `remake_nn.ipynb` (embeddings, input and
  prediction caps, early stopping on a time-ordered validation slice).
- **Hedonic** -- as in `remake_traindition.ipynb` (OLS on log price, one-hot
  categories, Duan smearing, input and prediction caps).

In [6]:
import time
import statsmodels.api as sm
import torch
from torch import nn
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


def rmse(y, yhat):
    return np.sqrt(np.mean((y - yhat) ** 2))


def mae(y, yhat):
    return np.mean(np.abs(y - yhat))


def mape(y, yhat):
    return np.mean(np.abs((y - yhat) / y)) * 100


TREE_MODELS = {
    "XGBoost": lambda: XGBRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    ),
    "CatBoost": lambda: CatBoostRegressor(
        iterations=500, depth=6, learning_rate=0.05, random_seed=42, thread_count=-1, verbose=0,
    ),
    "RandomForest": lambda: RandomForestRegressor(
        n_estimators=150, max_depth=18, min_samples_leaf=2, n_jobs=-1, random_state=42,
    ),
}


def run_tree(make_model, fold, cols):
    model = make_model()
    model.fit(fold["X_train"][cols], fold["y_train"])
    return model.predict(fold["X_test"][cols])


# ---------------- hedonic (as remake_traindition.ipynb) ----------------
def run_hedonic(fold, cat, boo, num):
    cols = cat + boo + num
    X_train, X_test = fold["X_train"][cols].copy(), fold["X_test"][cols].copy()
    low, high = X_train[num].quantile(0.005), X_train[num].quantile(0.995)
    X_train[num] = X_train[num].clip(low, high, axis=1)
    X_test[num] = X_test[num].clip(low, high, axis=1)

    def dummies(X):
        X = X.copy()
        X[cat] = X[cat].astype(int).astype(str)
        return pd.get_dummies(X, columns=cat, dtype=float)

    D = dummies(X_train)
    D = D.drop(columns=[f"{c}_{int(X_train[c].mode().iloc[0])}" for c in cat])
    D = D.loc[:, D.nunique() > 1]
    y = fold["y_train"]
    ols = sm.OLS(np.log(y), sm.add_constant(D, has_constant="add")).fit()
    smear = np.mean(np.exp(ols.resid))
    Dt = dummies(X_test).reindex(columns=D.columns, fill_value=0.0)
    pred = np.exp(np.asarray(ols.predict(sm.add_constant(Dt, has_constant="add")))) * smear
    return np.clip(pred, y.min(), y.max()), ols


# ---------------- neural network (as remake_nn.ipynb) ----------------
class PriceNet(nn.Module):
    def __init__(self, cardinalities, n_dense):
        super().__init__()
        self.embeddings = nn.ModuleList(nn.Embedding(c, min(16, (c + 1) // 2)) for c in cardinalities)
        emb_dim = sum(e.embedding_dim for e in self.embeddings)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + n_dense, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, cat, dense):
        emb = [e(cat[:, j]) for j, e in enumerate(self.embeddings)]
        return self.mlp(torch.cat(emb + [dense], dim=1)).squeeze(1)


def run_nn(fold, cat, boo, num, seed=42):
    dense_cols = boo + num
    X_train, X_test = fold["X_train"], fold["X_test"]
    cat_train = X_train[cat].to_numpy(dtype=np.int64) + 1
    cards = [int(cat_train[:, j].max()) + 1 for j in range(len(cat))]
    cat_test = np.clip(X_test[cat].to_numpy(dtype=np.int64) + 1, 0, np.array(cards) - 1)
    low, high = X_train[dense_cols].quantile(0.005), X_train[dense_cols].quantile(0.995)
    d_train = X_train[dense_cols].clip(low, high, axis=1)
    d_test = X_test[dense_cols].clip(low, high, axis=1)
    mean, std = d_train.mean(), d_train.std().replace(0, 1)
    d_train = ((d_train - mean) / std).to_numpy(dtype=np.float32)
    d_test = ((d_test - mean) / std).to_numpy(dtype=np.float32)
    log_y = np.log(fold["y_train"])
    y_mean, y_std = log_y.mean(), log_y.std()
    y_train = ((log_y - y_mean) / y_std).astype(np.float32)

    torch.manual_seed(seed)
    n = len(y_train)
    n_fit = n - int(n * 0.1)
    cat_t, dense_t, y_t = torch.from_numpy(cat_train), torch.from_numpy(d_train), torch.from_numpy(y_train)
    model = PriceNet(cards, d_train.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    best, best_state, bad = np.inf, None, 0
    for _ in range(100):
        model.train()
        order = torch.randperm(n_fit)
        for s in range(0, n_fit, 512):
            idx = order[s:s + 512]
            opt.zero_grad()
            loss_fn(model(cat_t[:n_fit][idx], dense_t[:n_fit][idx]), y_t[:n_fit][idx]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(model(cat_t[n_fit:], dense_t[n_fit:]), y_t[n_fit:]).item()
        if v < best - 1e-4:
            best, bad, best_state = v, 0, {k: t.clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= 10:
                break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        log_fit = model(cat_t, dense_t).numpy() * y_std + y_mean
        log_test = model(torch.from_numpy(cat_test), torch.from_numpy(d_test)).numpy() * y_std + y_mean
    smear = np.mean(np.exp(log_y - log_fit))
    return np.clip(np.exp(log_test) * smear, fold["y_train"].min(), fold["y_train"].max())

## 6. Train every model on every feature set

4 feature sets x 5 models x 5 folds. Takes a few minutes (mostly the
neural network).

In [7]:
actual = np.concatenate([f["y_test"] for f in folds])
rows = []
start_all = time.time()
for variant, drop in VARIANTS.items():
    cat, boo, num = groups(drop)
    cols = cat + boo + num
    preds = {m: [] for m in list(TREE_MODELS) + ["NeuralNet", "Hedonic"]}
    for fold in folds:
        for name, make_model in TREE_MODELS.items():
            preds[name].append(run_tree(make_model, fold, cols))
        preds["NeuralNet"].append(run_nn(fold, cat, boo, num))
        preds["Hedonic"].append(run_hedonic(fold, cat, boo, num)[0])
    for name, p in preds.items():
        p = np.concatenate(p)
        rows.append({"variant": variant, "model": name,
                     "RMSE": rmse(actual, p), "MAE": mae(actual, p), "MAPE": mape(actual, p)})
    print(f"{variant}: done ({time.time() - start_all:.0f}s so far)")

comparison = pd.DataFrame(rows)

A (29, all): done (104s so far)


B (27, drop quarter + building floor): done (222s so far)


C (27, drop quarter + mass floor): done (340s so far)


D (25, B + drop (un)employment): done (461s so far)


### 6.1 Accuracy by feature set

RMSE in dollars and MAPE in %, pooled over the same 48,777 test sales. The
"vs A" columns show the change from the full 29-feature set (negative = better).

In [8]:
model_order = ["XGBoost", "CatBoost", "RandomForest", "NeuralNet", "Hedonic"]
rmse_table = comparison.pivot(index="model", columns="variant", values="RMSE").loc[model_order]
mape_table = comparison.pivot(index="model", columns="variant", values="MAPE").loc[model_order]
base = list(VARIANTS)[0]

print("RMSE ($)")
display(rmse_table.round(0))
print("RMSE change vs A ($)")
display(rmse_table.sub(rmse_table[base], axis=0).drop(columns=base).round(0))
print("MAPE (%)")
display(mape_table.round(2))

RMSE ($)


variant,"A (29, all)","B (27, drop quarter + building floor)","C (27, drop quarter + mass floor)","D (25, B + drop (un)employment)"
model,,,,
XGBoost,212466.0,213178.0,212678.0,214007.0
CatBoost,219453.0,221538.0,220406.0,221722.0
RandomForest,227928.0,229112.0,228130.0,229537.0
NeuralNet,248911.0,248010.0,247526.0,251122.0
Hedonic,305066.0,307777.0,306564.0,319715.0


RMSE change vs A ($)


variant,"B (27, drop quarter + building floor)","C (27, drop quarter + mass floor)","D (25, B + drop (un)employment)"
model,,,
XGBoost,712.0,212.0,1542.0
CatBoost,2086.0,953.0,2269.0
RandomForest,1185.0,202.0,1609.0
NeuralNet,-901.0,-1385.0,2211.0
Hedonic,2711.0,1498.0,14649.0


MAPE (%)


variant,"A (29, all)","B (27, drop quarter + building floor)","C (27, drop quarter + mass floor)","D (25, B + drop (un)employment)"
model,,,,
XGBoost,10.95,11.01,10.98,11.01
CatBoost,11.60,11.73,11.61,11.72
RandomForest,11.57,11.70,11.55,11.70
NeuralNet,12.36,12.23,12.29,12.38
Hedonic,15.61,15.79,15.72,16.21


## 7. How much do the features overlap? (VIF)

The **variance inflation factor** measures how well each numeric feature
can be predicted from the others. VIF = 1 means no overlap; above about 10
is usually treated as a problem for a linear model, because its coefficient
becomes unstable. Computed on fold 5's training rows.

In [9]:
from statsmodels.stats.outliers_influence import variance_inflation_factor


def vif_table(drop):
    _, boo, num = groups(drop)
    X = folds[-1]["X_train"][boo + num]
    X = X.loc[:, X.std() > 0]
    Xc = sm.add_constant(X, has_constant="add").to_numpy()
    return pd.Series([variance_inflation_factor(Xc, j + 1) for j in range(X.shape[1])], index=X.columns)


vif = pd.DataFrame({name: vif_table(drop) for name, drop in VARIANTS.items()})
vif.sort_values(base, ascending=False).round(1)

,"A (29, all)","B (27, drop quarter + building floor)","C (27, drop quarter + mass floor)","D (25, B + drop (un)employment)"
building_total_floor_area,18.2,NaN,1.4,NaN
sale_month,18.0,1.0,1.0,1.0
sale_quarter,18.0,NaN,NaN,NaN
mass_total_floor_area,17.4,1.3,NaN,1.3
median_personal_income_by_sa,7.3,7.3,7.3,2.8
area_level_deprivation_index,5.4,5.4,5.4,3.5
employment_rate_by_sa,4.6,4.6,4.6,NaN
unemployment_rate_by_sa,3.5,3.5,3.5,NaN
population_density,2.4,2.4,2.4,2.4
latitude,2.1,2.1,2.1,2.0


## 8. Do the hedonic coefficients make more sense?

With overlapping features a linear model can give a coefficient the "wrong"
sign (e.g. a higher employment rate lowering price) because a correlated
feature already carries that information. Here: the census / deprivation
coefficients on fold 5, as a % change in price per step, for the full set A
and the reduced set D.

In [10]:
STEPS = {
    "area_level_deprivation_index": (1, "per decile"),
    "median_personal_income_by_sa": (10_000, "per $10K"),
    "employment_rate_by_sa": (1, "per percentage point"),
    "unemployment_rate_by_sa": (1, "per percentage point"),
    "building_total_floor_area": (np.log(1.01), "per 1% more"),
    "mass_total_floor_area": (np.log(1.01), "per 1% more"),
}
coef_rows = {}
for name in [base, list(VARIANTS)[3]]:
    _, ols = run_hedonic(folds[-1], *groups(VARIANTS[name]))
    coef_rows[name] = {
        f: ((np.exp(ols.params[f] * step) - 1) * 100 if f in ols.params else np.nan)
        for f, (step, _) in STEPS.items()
    }
coef_compare = pd.DataFrame(coef_rows)
coef_compare.insert(0, "step", [label for _, label in STEPS.values()])
coef_compare.round(2)

,step,"A (29, all)","D (25, B + drop (un)employment)"
area_level_deprivation_index,per decile,-1.23,-2.84
median_personal_income_by_sa,per $10K,12.21,-0.14
employment_rate_by_sa,per percentage point,-1.70,NaN
unemployment_rate_by_sa,per percentage point,-1.39,NaN
building_total_floor_area,per 1% more,0.30,NaN
mass_total_floor_area,per 1% more,0.18,0.44


## 9. SHAP: full set A vs reduced set D (XGBoost)

Fold-5 XGBoost, same 500 test sales as the other notebooks. Shares of total
mean |SHAP|. In set D the credit no longer has to be split between
near-duplicate columns.

In [11]:
import shap

shap_fold = folds[-1]
shap_sample = shap_fold["X_test"].sample(n=500, random_state=42)
shares = {}
for name in [base, list(VARIANTS)[3]]:
    cat, boo, num = groups(VARIANTS[name])
    cols = cat + boo + num
    model = TREE_MODELS["XGBoost"]()
    model.fit(shap_fold["X_train"][cols], shap_fold["y_train"])
    sv = shap.TreeExplainer(model).shap_values(shap_sample[cols])
    s = pd.Series(np.abs(sv).mean(axis=0), index=cols)
    shares[name] = s / s.sum() * 100

shap_compare = pd.DataFrame(shares)
shap_compare.sort_values(base, ascending=False).round(1).head(15)

D:\Codes\COMPSCI_760_Group_3_Intelligent_Real_Estate_Price_Prediction_Model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,"A (29, all)","D (25, B + drop (un)employment)"
distance_to_cbd,16.5,16.700001
mass_total_floor_area,13.4,20.600000
land_area,12.6,13.300000
building_total_floor_area,10.2,NaN
longitude,8.3,9.600000
latitude,8.0,8.600000
area_level_deprivation_index,5.2,7.300000
building_age_indicator,4.6,4.600000
median_personal_income_by_sa,3.0,1.800000
employment_rate_by_sa,3.0,NaN
